# ORBIS/Aurelia Import Inspection

Safe local inspection notebook for NDA-restricted ORBIS/Aurelia exports. The cells below avoid printing firm-level records and only show file metadata, headers, row counts, and aggregate URL/domain diagnostics.

Before committing any notebook changes, clear all outputs.

In [ ]:
from pathlib import Path
import csv
import io
from urllib.parse import urlsplit

PROJECT_ROOT = Path.cwd()
INPUT_FILES = [
    PROJECT_ROOT / "data/raw/orbis_aurelia/Aurelia_Export_1.txt",
    PROJECT_ROOT / "data/raw/orbis_aurelia/Aurelia_Export_2.txt",
]

REQUIRED_COLUMNS = [
    "Mark",
    "Name des Unternehmens",
    "BvD ID Nummer",
    "Postleitzahl",
    "Ort",
    "ONACE 2008 Haupttätigkeit 4-stellig - Code",
    "Web Adresse",
    "E-Mail Adresse",
]

In [ ]:
def read_text_with_diagnostics(path: Path):
    encodings = ["utf-8-sig", "utf-16", "utf-16-le", "cp1252", "latin-1"]
    errors = {}
    for encoding in encodings:
        try:
            return path.read_text(encoding=encoding), encoding, errors
        except UnicodeError as exc:
            errors[encoding] = type(exc).__name__
    raise UnicodeError(f"Could not decode {path}")


def sniff_file(path: Path):
    text, encoding, errors = read_text_with_diagnostics(path)
    reader = csv.reader(io.StringIO(text), delimiter=";", quotechar='"')
    header = next(reader)
    row_count = sum(1 for _ in reader)
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(header))
    return {
        "file": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2),
        "encoding_used": encoding,
        "failed_encodings": errors,
        "row_count_excluding_header": row_count,
        "column_count": len(header),
        "missing_required_columns": missing_columns,
        "header": header,
    }


for input_file in INPUT_FILES:
    sniff_file(input_file)

In [ ]:
def normalize_domain(raw_url):
    if raw_url is None or not raw_url.strip():
        return None, "missing"
    url = raw_url.strip()
    if any(char.isspace() for char in url):
        return None, "invalid"
    if "://" not in url:
        url = f"https://{url}"
    parsed = urlsplit(url)
    if parsed.scheme.lower() not in {"http", "https"} or not parsed.hostname:
        return None, "invalid"
    domain = parsed.hostname.lower().removeprefix("www.")
    if "." not in domain:
        return None, "invalid"
    return domain, "valid"


def aggregate_diagnostics(path: Path):
    text, encoding, _errors = read_text_with_diagnostics(path)
    reader = csv.DictReader(io.StringIO(text), delimiter=";", quotechar='"')
    bvd_ids = set()
    duplicate_bvd_ids = set()
    domains = set()
    missing_urls = 0
    invalid_urls = 0
    rows = 0
    for row in reader:
        rows += 1
        bvd_id = (row.get("BvD ID Nummer") or "").strip()
        if bvd_id in bvd_ids:
            duplicate_bvd_ids.add(bvd_id)
        elif bvd_id:
            bvd_ids.add(bvd_id)

        domain, status = normalize_domain(row.get("Web Adresse"))
        if status == "missing":
            missing_urls += 1
        elif status == "invalid":
            invalid_urls += 1
        elif domain:
            domains.add(domain)

    return {
        "file": str(path.relative_to(PROJECT_ROOT)),
        "encoding_used": encoding,
        "rows": rows,
        "unique_bvd_ids": len(bvd_ids),
        "duplicate_bvd_id_count": len(duplicate_bvd_ids),
        "unique_valid_domains": len(domains),
        "missing_urls": missing_urls,
        "invalid_urls": invalid_urls,
    }


for input_file in INPUT_FILES:
    aggregate_diagnostics(input_file)